<a href="https://colab.research.google.com/github/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/blob/main/AgenticAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Math

In [7]:
from google.colab import userdata
from huggingface_hub import login
import os
import sys
import time

In [3]:
HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

In [6]:
repo_url = "https://github.com/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi.git"
repo_name = "NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi"

if os.path.exists("../"+repo_name):
    print("Repository already present, update...")
    !git pull
else:
    print("Repository clone...")
    !git clone {repo_url}
    %cd {repo_name}

sys.path.append('/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/NLP_assignment_api_client')

from millionaire_client import MillionaireClient, AuthenticationError, GameError

Repository clone...
Cloning into 'NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi'...
remote: Enumerating objects: 257, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 257 (delta 50), reused 14 (delta 4), pack-reused 168 (from 1)
Receiving objects: 100% (257/257), 930.18 KiB | 1.12 MiB/s, done.
Resolving deltas: 100% (129/129), done.
/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi


NameError: name 'sys' is not defined

# Math finetuning

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig , TrainingArguments, pipeline, TextStreamer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from tqdm import tqdm

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Thinking-2507",
    max_seq_length = 2048,
    load_in_4bit = True                         # 4 bit quantization to reduce memory
)

==((====))==  Unsloth 2026.5.7: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen3-4b-thinking-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,                                     # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,                           # Supports any, but = 0 is optimized
    bias = "none",                              # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth",     # True or "unsloth" for very long context
    random_state = 3407
)
tokenizer = get_chat_template(tokenizer, chat_template = "qwen3-thinking")

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

# Tools

In [ ]:
import math
from langchain.tools import tool
from sympy import sympify, Eq, solve
import re


In [ ]:
@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calculator(expression: str) -> str:
    """
    Evaluate mathematical expressions.
    Parameters: expression str type containing arithmetic python code
    """
    return str(eval(expression))

In [ ]:
@tool
def solve_equation(expressions):
      """
      Solve a mathematical equation or a system of mathematical equations.
      Parameters: expressions array of str types containing the equation or equations to solve.
      """

      expressions = [expressions]

      equations = []
      symbols_set = set()

      variable_names = set()

      for expr in expressions:
          variable_names.update(
              re.findall(r"[a-zA-Z_]\w*", expr)
          )

      symbols_dict = {
          name: symbols(name)
          for name in variable_names
      }

      for expr in expressions:

          if "=" in expr:

              left, right = expr.split("=")

              eq = Eq(
                  eval(left, {}, symbols_dict),
                  eval(right, {}, symbols_dict)
              )

          else:

              eq = Eq(
                  eval(expr, {}, symbols_dict),
                  0
              )

          equations.append(eq)

          symbols_set.update(eq.free_symbols)

      variables = list(symbols_set)

      result = solve(equations, variables)

      return str(result)